
# Roxy notebook example: Hybrid descriptors aggregated by property families

This notebook is a **reference implementation example** for the **hybrid descriptor family aggregated by property groups** in Roxy.

This family is designed to summarize multiple descriptor spaces into a **smaller, interpretable set of family-level features**. Instead of exposing every raw descriptor independently, it aggregates them into biologically meaningful blocks such as:

- charge-related properties
- hydrophobicity and polarity
- order/disorder tendencies
- functional residue content
- structural propensity
- repetition and complexity
- positional and regional organization

These descriptors are useful when the user wants:

- a compact feature table
- interpretable summaries for EDA or ML baselines
- lower-dimensional hand-crafted representations
- family-level biological characterization rather than raw granular features

## Covered outputs

This notebook implements examples such as:

- family-level aggregated means
- family-level aggregated amplitudes
- family-level burden scores
- family-level balance scores
- family-level contrast between related properties
- compact descriptor tables derived from multiple property sources
- class-style implementation for later migration into Roxy

The notebook is designed as a **clean teaching implementation** so it can later become part of the real Roxy package.


In [1]:

from collections import Counter
from itertools import groupby

import numpy as np
import pandas as pd


## Demo dataset

In [2]:

df_demo = pd.DataFrame(
    {
        "sequence_id": [
            "hyb_1",
            "hyb_2",
            "hyb_3",
            "hyb_4",
            "hyb_5",
            "hyb_6",
        ],
        "sequence": [
            "MKWVTFISLLFLFSSAYSRGVFRR",
            "GGGGGGGGGGGGGGG",
            "KRRKRRKRRKRRDDDDEE",
            "ACDEFGHIKLMNPQRSTVWY",
            "PPPPGSSSSSTTTTNNQQQ",
            "MSTNPKPQRITLKDGNKVELV",
        ],
        "label": ["A", "B", "A", "B", "A", "B"],
    }
)

df_demo


,sequence_id,sequence,label
0,hyb_1,MKWVTFISLLFLFSSAYSRGVFRR,A
1,hyb_2,GGGGGGGGGGGGGGG,B
2,hyb_3,KRRKRRKRRKRRDDDDEE,A
3,hyb_4,ACDEFGHIKLMNPQRSTVWY,B
4,hyb_5,PPPPGSSSSSTTTTNNQQQ,A
5,hyb_6,MSTNPKPQRITLKDGNKVELV,B


## Constants

In [3]:

STANDARD_AA = set("ACDEFGHIKLMNPQRSTVWY")

HYDROPATHY = {
    "A": 1.8, "C": 2.5, "D": -3.5, "E": -3.5, "F": 2.8,
    "G": -0.4, "H": -3.2, "I": 4.5, "K": -3.9, "L": 3.8,
    "M": 1.9, "N": -3.5, "P": -1.6, "Q": -3.5, "R": -4.5,
    "S": -0.8, "T": -0.7, "V": 4.2, "W": -0.9, "Y": -1.3,
}

POLARITY = {
    "A": 8.1, "C": 5.5, "D": 13.0, "E": 12.3, "F": 5.2,
    "G": 9.0, "H": 10.4, "I": 5.2, "K": 11.3, "L": 4.9,
    "M": 5.7, "N": 11.6, "P": 8.0, "Q": 10.5, "R": 10.5,
    "S": 9.2, "T": 8.6, "V": 5.9, "W": 5.4, "Y": 6.2,
}

HELIX_PROP = {
    "A": 1.45, "C": 0.77, "D": 1.01, "E": 1.53, "F": 1.12,
    "G": 0.53, "H": 1.24, "I": 1.00, "K": 1.07, "L": 1.34,
    "M": 1.20, "N": 0.73, "P": 0.59, "Q": 1.17, "R": 0.79,
    "S": 0.79, "T": 0.82, "V": 1.14, "W": 1.14, "Y": 0.61,
}

SHEET_PROP = {
    "A": 0.97, "C": 1.30, "D": 0.54, "E": 0.37, "F": 1.28,
    "G": 0.81, "H": 0.71, "I": 1.60, "K": 0.74, "L": 1.22,
    "M": 1.67, "N": 0.65, "P": 0.62, "Q": 1.23, "R": 0.90,
    "S": 0.72, "T": 1.20, "V": 1.65, "W": 1.19, "Y": 1.29,
}

TURN_PROP = {
    "A": 0.66, "C": 1.19, "D": 1.46, "E": 0.74, "F": 0.60,
    "G": 1.56, "H": 0.95, "I": 0.47, "K": 1.01, "L": 0.59,
    "M": 0.60, "N": 1.56, "P": 1.52, "Q": 0.98, "R": 0.95,
    "S": 1.43, "T": 0.96, "V": 0.50, "W": 0.96, "Y": 1.14,
}

AA_GROUPS = {
    "positive": set("KRH"),
    "negative": set("DE"),
    "charged": set("KRHDE"),
    "polar": set("STNQCYWHKRDE"),
    "nonpolar": set("AVLIMFGP"),
    "aromatic": set("FWYH"),
    "hydrophobic": set("AVLIMFWCY"),
    "hydrophilic": set("RNDQEHKST"),
    "disorder_promoting": set("ARGQSEPK"),
    "order_promoting": set("CWYFILNV"),
    "sulfur": set("CM"),
    "hydroxyl": set("STY"),
    "amide": set("NQ"),
}


## Helper functions

In [4]:

def clean_sequence(seq: str) -> str:
    if pd.isna(seq):
        return ""
    seq = str(seq).strip().upper().replace("*", "")
    return "".join([aa for aa in seq if aa in STANDARD_AA])


def scale_mean(seq: str, scale: dict) -> float:
    if len(seq) == 0:
        return np.nan
    return float(np.mean([scale[aa] for aa in seq]))


def scale_std(seq: str, scale: dict) -> float:
    if len(seq) == 0:
        return np.nan
    return float(np.std([scale[aa] for aa in seq], ddof=0))


def fraction_from_group(seq: str, aa_group) -> float:
    if len(seq) == 0:
        return np.nan
    return sum(aa in aa_group for aa in seq) / len(seq)


def shannon_entropy(seq: str) -> float:
    if len(seq) == 0:
        return np.nan
    counts = Counter(seq)
    probs = np.array([count / len(seq) for count in counts.values()], dtype=float)
    return float(-(probs * np.log2(probs)).sum())


def longest_homopolymer_run(seq: str) -> int:
    if len(seq) == 0:
        return 0
    return max(len(list(group)) for _, group in groupby(seq))


def windows(seq: str, size: int):
    if len(seq) < size:
        return []
    return [seq[i:i+size] for i in range(len(seq) - size + 1)]


def mean_local_charge_proxy(seq: str, window: int = 5) -> float:
    ws = windows(seq, window)
    if len(ws) == 0:
        return np.nan
    vals = []
    for w in ws:
        pos = fraction_from_group(w, AA_GROUPS["positive"])
        neg = fraction_from_group(w, AA_GROUPS["negative"])
        vals.append(pos - neg)
    return float(np.mean(vals))


def amplitude_local_charge_proxy(seq: str, window: int = 5) -> float:
    ws = windows(seq, window)
    if len(ws) == 0:
        return np.nan
    vals = []
    for w in ws:
        pos = fraction_from_group(w, AA_GROUPS["positive"])
        neg = fraction_from_group(w, AA_GROUPS["negative"])
        vals.append(pos - neg)
    return float(np.max(vals) - np.min(vals))


## Core hybrid family-level descriptor function

In [5]:

def hybrid_family_descriptors(seq: str) -> dict:
    seq = clean_sequence(seq)

    out = {
        "hyb_length": len(seq),
        "hyb_valid_residue_count": len(seq),
    }

    if len(seq) == 0:
        return out

    # Charge family
    pos_frac = fraction_from_group(seq, AA_GROUPS["positive"])
    neg_frac = fraction_from_group(seq, AA_GROUPS["negative"])
    charged_frac = fraction_from_group(seq, AA_GROUPS["charged"])
    out["hyb_charge_family_mean"] = float(np.mean([pos_frac, neg_frac, charged_frac]))
    out["hyb_charge_family_balance"] = pos_frac - neg_frac
    out["hyb_charge_family_local_mean"] = mean_local_charge_proxy(seq, window=5)
    out["hyb_charge_family_local_amplitude"] = amplitude_local_charge_proxy(seq, window=5)

    # Hydropathy / polarity family
    hydropathy_mean = scale_mean(seq, HYDROPATHY)
    hydropathy_std = scale_std(seq, HYDROPATHY)
    polarity_mean = scale_mean(seq, POLARITY)
    polarity_std = scale_std(seq, POLARITY)
    hydrophobic_frac = fraction_from_group(seq, AA_GROUPS["hydrophobic"])
    hydrophilic_frac = fraction_from_group(seq, AA_GROUPS["hydrophilic"])
    polar_frac = fraction_from_group(seq, AA_GROUPS["polar"])
    nonpolar_frac = fraction_from_group(seq, AA_GROUPS["nonpolar"])

    out["hyb_physchem_family_mean"] = float(np.mean([hydropathy_mean, polarity_mean]))
    out["hyb_physchem_family_dispersion"] = float(np.mean([hydropathy_std, polarity_std]))
    out["hyb_physchem_hydrophobic_balance"] = hydrophobic_frac - hydrophilic_frac
    out["hyb_physchem_polar_balance"] = polar_frac - nonpolar_frac

    # Structure-propensity family
    helix_mean = scale_mean(seq, HELIX_PROP)
    sheet_mean = scale_mean(seq, SHEET_PROP)
    turn_mean = scale_mean(seq, TURN_PROP)
    out["hyb_struct_family_mean"] = float(np.mean([helix_mean, sheet_mean, turn_mean]))
    out["hyb_struct_helix_sheet_balance"] = helix_mean - sheet_mean
    out["hyb_struct_turn_secondary_balance"] = turn_mean - ((helix_mean + sheet_mean) / 2.0)

    # Order/disorder family
    disorder_frac = fraction_from_group(seq, AA_GROUPS["disorder_promoting"])
    order_frac = fraction_from_group(seq, AA_GROUPS["order_promoting"])
    out["hyb_orderdis_family_mean"] = float(np.mean([disorder_frac, order_frac]))
    out["hyb_orderdis_balance"] = disorder_frac - order_frac

    # Functional residue family
    sulfur_frac = fraction_from_group(seq, AA_GROUPS["sulfur"])
    hydroxyl_frac = fraction_from_group(seq, AA_GROUPS["hydroxyl"])
    amide_frac = fraction_from_group(seq, AA_GROUPS["amide"])
    aromatic_frac = fraction_from_group(seq, AA_GROUPS["aromatic"])
    out["hyb_functional_family_mean"] = float(np.mean([sulfur_frac, hydroxyl_frac, amide_frac, aromatic_frac]))
    out["hyb_functional_reactivity_proxy"] = float(np.mean([sulfur_frac, hydroxyl_frac, pos_frac, neg_frac]))

    # Complexity / repetition family
    entropy = shannon_entropy(seq)
    longest_homo = longest_homopolymer_run(seq)
    repeated_burden = longest_homo / len(seq)
    out["hyb_complexity_family_mean"] = float(np.mean([entropy, repeated_burden]))
    out["hyb_complexity_entropy"] = entropy
    out["hyb_complexity_repeat_burden"] = repeated_burden

    # Compact global hybrid summaries
    family_core_values = [
        out["hyb_charge_family_mean"],
        out["hyb_physchem_family_mean"],
        out["hyb_struct_family_mean"],
        out["hyb_orderdis_family_mean"],
        out["hyb_functional_family_mean"],
        out["hyb_complexity_family_mean"],
    ]

    out["hyb_global_family_mean"] = float(np.nanmean(family_core_values))
    out["hyb_global_family_std"] = float(np.nanstd(family_core_values, ddof=0))
    out["hyb_global_family_max"] = float(np.nanmax(family_core_values))
    out["hyb_global_family_min"] = float(np.nanmin(family_core_values))
    out["hyb_global_family_amplitude"] = out["hyb_global_family_max"] - out["hyb_global_family_min"]

    return out


## Functional usage on one sequence

In [6]:

example = hybrid_family_descriptors(df_demo.loc[0, "sequence"])
list(example.items())[:20]


[('hyb_length', 24),
 ('hyb_valid_residue_count', 24),
 ('hyb_charge_family_mean', 0.1111111111111111),
 ('hyb_charge_family_balance', 0.16666666666666666),
 ('hyb_charge_family_local_mean', 0.1),
 ('hyb_charge_family_local_amplitude', 0.4),
 ('hyb_physchem_family_mean', 3.9666666666666672),
 ('hyb_physchem_family_dispersion', 2.579425750001655),
 ('hyb_physchem_hydrophobic_balance', 0.20833333333333337),
 ('hyb_physchem_polar_balance', -0.08333333333333331),
 ('hyb_struct_family_mean', 1.0050000000000001),
 ('hyb_struct_helix_sheet_balance', -0.125),
 ('hyb_struct_turn_secondary_balance', -0.1887500000000003),
 ('hyb_orderdis_family_mean', 0.45833333333333337),
 ('hyb_orderdis_balance', -0.08333333333333331),
 ('hyb_functional_family_mean', 0.13541666666666669),
 ('hyb_functional_reactivity_proxy', 0.11458333333333334),
 ('hyb_complexity_family_mean', 1.7610276044371003),
 ('hyb_complexity_entropy', 3.438721875540867),
 ('hyb_complexity_repeat_burden', 0.08333333333333333)]

## Apply hybrid family descriptors to the full dataset

In [7]:

df_hyb = pd.concat(
    [
        df_demo,
        df_demo["sequence"].apply(hybrid_family_descriptors).apply(pd.Series),
    ],
    axis=1,
)

df_hyb.head()


,sequence_id,sequence,label,hyb_length,hyb_valid_residue_count,hyb_charge_family_mean,hyb_charge_family_balance,hyb_charge_family_local_mean,hyb_charge_family_local_amplitude,hyb_physchem_family_mean,...,hyb_functional_family_mean,hyb_functional_reactivity_proxy,hyb_complexity_family_mean,hyb_complexity_entropy,hyb_complexity_repeat_burden,hyb_global_family_mean,hyb_global_family_std,hyb_global_family_max,hyb_global_family_min,hyb_global_family_amplitude
0,hyb_1,MKWVTFISLLFLFSSAYSRGVFRR,A,24.0,24.0,0.111111,0.166667,0.100000,0.4,3.966667,...,0.135417,0.114583,1.761028,3.438722,0.083333,1.239593,1.345790,3.966667,0.111111,3.855556
1,hyb_2,GGGGGGGGGGGGGGG,B,15.0,15.0,0.000000,0.000000,0.000000,0.0,4.300000,...,0.000000,0.000000,0.500000,-0.000000,1.000000,1.044444,1.493277,4.300000,0.000000,4.300000
2,hyb_3,KRRKRRKRRKRRDDDDEE,A,18.0,18.0,0.666667,0.333333,0.428571,2.0,3.700000,...,0.000000,0.250000,1.029407,1.836592,0.222222,1.117617,1.204070,3.700000,0.000000,3.700000
3,hyb_4,ACDEFGHIKLMNPQRSTVWY,B,20.0,20.0,0.166667,0.050000,0.100000,0.8,3.917500,...,0.137500,0.125000,2.185964,4.321928,0.050000,1.302744,1.364235,3.917500,0.137500,3.780000
4,hyb_5,PPPPGSSSSSTTTTNNQQQ,A,19.0,19.0,0.000000,0.000000,0.000000,0.0,3.815789,...,0.184211,0.118421,1.351213,2.439268,0.263158,1.122746,1.290579,3.815789,0.000000,3.815789


## Inspect descriptor columns

In [8]:

hyb_cols = [c for c in df_hyb.columns if c.startswith("hyb_") and c not in {"hyb_length", "hyb_valid_residue_count"}]
len(hyb_cols), hyb_cols[:18]


(23,
 ['hyb_charge_family_mean',
  'hyb_charge_family_balance',
  'hyb_charge_family_local_mean',
  'hyb_charge_family_local_amplitude',
  'hyb_physchem_family_mean',
  'hyb_physchem_family_dispersion',
  'hyb_physchem_hydrophobic_balance',
  'hyb_physchem_polar_balance',
  'hyb_struct_family_mean',
  'hyb_struct_helix_sheet_balance',
  'hyb_struct_turn_secondary_balance',
  'hyb_orderdis_family_mean',
  'hyb_orderdis_balance',
  'hyb_functional_family_mean',
  'hyb_functional_reactivity_proxy',
  'hyb_complexity_family_mean',
  'hyb_complexity_entropy',
  'hyb_complexity_repeat_burden'])

In [9]:

df_hyb[
    [
        "sequence_id",
        "hyb_charge_family_mean",
        "hyb_physchem_family_mean",
        "hyb_struct_family_mean",
        "hyb_orderdis_balance",
        "hyb_functional_reactivity_proxy",
        "hyb_complexity_entropy",
        "hyb_global_family_mean",
        "hyb_global_family_amplitude",
    ]
]


,sequence_id,hyb_charge_family_mean,hyb_physchem_family_mean,hyb_struct_family_mean,hyb_orderdis_balance,hyb_functional_reactivity_proxy,hyb_complexity_entropy,hyb_global_family_mean,hyb_global_family_amplitude
0,hyb_1,0.111111,3.966667,1.005000,-0.083333,0.114583,3.438722,1.239593,3.855556
1,hyb_2,0.000000,4.300000,0.966667,1.000000,0.000000,-0.000000,1.044444,4.300000
2,hyb_3,0.666667,3.700000,0.920741,0.777778,0.250000,1.836592,1.117617,3.700000
3,hyb_4,0.166667,3.917500,1.008833,0.000000,0.125000,4.321928,1.302744,3.780000
4,hyb_5,0.000000,3.815789,0.990526,0.578947,0.118421,2.439268,1.122746,3.815789
5,hyb_6,0.190476,4.052381,0.995079,0.142857,0.119048,3.689704,1.265782,3.969048


## Dataset-level summary

In [10]:

hyb_summary = (
    df_hyb[hyb_cols]
    .mean(axis=0, numeric_only=True)
    .sort_values(ascending=False)
    .rename("mean_value")
    .reset_index()
    .rename(columns={"index": "descriptor"})
)

hyb_summary.head(15)


,descriptor,mean_value
0,hyb_physchem_family_mean,3.958723
1,hyb_global_family_max,3.958723
2,hyb_global_family_amplitude,3.903399
3,hyb_complexity_entropy,2.621036
4,hyb_physchem_family_dispersion,1.680926
5,hyb_complexity_family_mean,1.449379
6,hyb_global_family_std,1.347042
7,hyb_global_family_mean,1.182154
8,hyb_struct_family_mean,0.981141
9,hyb_charge_family_local_amplitude,0.600000


## Sanity checks

In [11]:

assert "hyb_charge_family_mean" in df_hyb.columns
assert "hyb_physchem_family_mean" in df_hyb.columns
assert "hyb_struct_family_mean" in df_hyb.columns
assert "hyb_orderdis_balance" in df_hyb.columns
assert "hyb_functional_reactivity_proxy" in df_hyb.columns
assert "hyb_global_family_amplitude" in df_hyb.columns
assert df_hyb["hyb_length"].min() > 0

print(f"Number of hybrid aggregated descriptor columns: {len(hyb_cols)}")
print("Hybrid family descriptor checks passed.")


Number of hybrid aggregated descriptor columns: 23
Hybrid family descriptor checks passed.


## Class-style implementation closer to the real package

In [12]:

class HybridFamilyDescriptors:
    """Example class-style hybrid aggregated implementation for later migration into Roxy."""

    def transform_sequence(self, seq: str) -> dict:
        return hybrid_family_descriptors(seq)

    def transform(self, sequences) -> pd.DataFrame:
        return pd.DataFrame([self.transform_sequence(seq) for seq in sequences])


hyb_transformer = HybridFamilyDescriptors()
hyb_matrix = hyb_transformer.transform(df_demo["sequence"].tolist())
hyb_matrix.head()


,hyb_length,hyb_valid_residue_count,hyb_charge_family_mean,hyb_charge_family_balance,hyb_charge_family_local_mean,hyb_charge_family_local_amplitude,hyb_physchem_family_mean,hyb_physchem_family_dispersion,hyb_physchem_hydrophobic_balance,hyb_physchem_polar_balance,...,hyb_functional_family_mean,hyb_functional_reactivity_proxy,hyb_complexity_family_mean,hyb_complexity_entropy,hyb_complexity_repeat_burden,hyb_global_family_mean,hyb_global_family_std,hyb_global_family_max,hyb_global_family_min,hyb_global_family_amplitude
0,24,24,0.111111,0.166667,0.100000,0.4,3.966667,2.579426e+00,0.208333,-0.083333,...,0.135417,0.114583,1.761028,3.438722,0.083333,1.239593,1.345790,3.966667,0.111111,3.855556
1,15,15,0.000000,0.000000,0.000000,0.0,4.300000,5.551115e-17,0.000000,-1.000000,...,0.000000,0.000000,0.500000,-0.000000,1.000000,1.044444,1.493277,4.300000,0.000000,4.300000
2,18,18,0.666667,0.333333,0.428571,2.0,3.700000,7.260836e-01,-1.000000,1.000000,...,0.000000,0.250000,1.029407,1.836592,0.222222,1.117617,1.204070,3.700000,0.000000,3.700000
3,20,20,0.166667,0.050000,0.100000,0.8,3.917500,2.766860e+00,0.000000,0.200000,...,0.137500,0.125000,2.185964,4.321928,0.050000,1.302744,1.364235,3.917500,0.137500,3.780000
4,19,19,0.000000,0.000000,0.000000,0.0,3.815789,1.142731e+00,-0.736842,0.473684,...,0.184211,0.118421,1.351213,2.439268,0.263158,1.122746,1.290579,3.815789,0.000000,3.815789


## Merge transformer output back to the dataset

In [13]:

df_hyb_class = pd.concat([df_demo, hyb_matrix], axis=1)
df_hyb_class.head()


,sequence_id,sequence,label,hyb_length,hyb_valid_residue_count,hyb_charge_family_mean,hyb_charge_family_balance,hyb_charge_family_local_mean,hyb_charge_family_local_amplitude,hyb_physchem_family_mean,...,hyb_functional_family_mean,hyb_functional_reactivity_proxy,hyb_complexity_family_mean,hyb_complexity_entropy,hyb_complexity_repeat_burden,hyb_global_family_mean,hyb_global_family_std,hyb_global_family_max,hyb_global_family_min,hyb_global_family_amplitude
0,hyb_1,MKWVTFISLLFLFSSAYSRGVFRR,A,24,24,0.111111,0.166667,0.100000,0.4,3.966667,...,0.135417,0.114583,1.761028,3.438722,0.083333,1.239593,1.345790,3.966667,0.111111,3.855556
1,hyb_2,GGGGGGGGGGGGGGG,B,15,15,0.000000,0.000000,0.000000,0.0,4.300000,...,0.000000,0.000000,0.500000,-0.000000,1.000000,1.044444,1.493277,4.300000,0.000000,4.300000
2,hyb_3,KRRKRRKRRKRRDDDDEE,A,18,18,0.666667,0.333333,0.428571,2.0,3.700000,...,0.000000,0.250000,1.029407,1.836592,0.222222,1.117617,1.204070,3.700000,0.000000,3.700000
3,hyb_4,ACDEFGHIKLMNPQRSTVWY,B,20,20,0.166667,0.050000,0.100000,0.8,3.917500,...,0.137500,0.125000,2.185964,4.321928,0.050000,1.302744,1.364235,3.917500,0.137500,3.780000
4,hyb_5,PPPPGSSSSSTTTTNNQQQ,A,19,19,0.000000,0.000000,0.000000,0.0,3.815789,...,0.184211,0.118421,1.351213,2.439268,0.263158,1.122746,1.290579,3.815789,0.000000,3.815789



## Suggested next refactor into the package

A clean migration path into Roxy would be:

- move family definitions and scales into `roxy/core/constants.py`
- move helper logic into `roxy/sequence/hybrid.py`
- expose a class such as `HybridFamilyDescriptors`
- allow configurable:
  - which families to aggregate
  - which base summaries define each family
  - whether to return only family-level features or both family and raw summaries
- add tests for:
  - empty sequences
  - strongly charge-biased sequences
  - strongly disorder-promoting sequences
  - repetitive low-complexity sequences
  - lower-case input
  - invalid characters removed during cleaning


## Optional export

In [14]:
# df_hyb.to_csv("demo_hybrid_family_descriptors.csv", index=False)
